# Glyph — Instruction-tuning LoRA (E5_instruct) — Kaggle версия

Обучает 3 LoRA-адаптера `E5_instruct` (Достоевский, Чехов, Булгаков) на synthetic QA-датасете.

## Настройки Kaggle

Перед запуском справа в панели Settings:
- **Accelerator:** `GPU T4 x2` или `GPU P100`
- **Internet:** `ON` (для git clone и pip install)
- **Persistence:** `Files only` (опционально — сохранит модели между сессиями)

После запуска «Save & Run All» Kaggle выполняет ноутбук **в фоне** — вкладку можно закрыть.

## Оценка времени
3 адаптера × 500 пар × 5 эпох ≈ **45-60 минут на T4**.

## 1. Проверка GPU

In [ ]:
!nvidia-smi | head -15
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Клонирование репозитория

Репо уже содержит synthetic-датасет и код — ничего загружать вручную не нужно.

In [ ]:
import os
os.chdir('/kaggle/working')
!rm -rf glyph
!git clone --depth 1 https://github.com/tearfu1/glyph.git
os.chdir('/kaggle/working/glyph/ml')
!ls data/synthetic/

## 3. Проверка synthetic-датасета

In [ ]:
import json
for author in ['dostoevsky', 'chekhov', 'bulgakov']:
    path = f'data/synthetic/{author}.jsonl'
    with open(path) as f:
        records = [json.loads(l) for l in f]
    print(f'{author}: {len(records)} пар')
    # одна случайная пара для проверки
    r = records[0]
    print(f'  Q ({len(r["question"].split())} слов): {r["question"][:100]}...')
    print(f'  A ({len(r["answer"].split())} слов): {r["answer"][:100]}...\n')

## 4. Установка зависимостей

Kaggle уже имеет torch+CUDA; устанавливаем остальное.

In [ ]:
!pip install -q transformers==4.51.0 peft==0.18.1 datasets==2.19.0 accelerate==0.30.0

# Форсируем переустановку accelerate (у Kaggle предустановлена старая версия)
!pip install -q --upgrade --force-reinstall 'accelerate>=0.33.0'
!pip install -q --upgrade transformers==4.51.0 peft==0.18.1 datasets==2.19.0

print("--- Проверка версий ---")
!pip show accelerate | grep -E "^(Name|Version)"
!pip show peft | grep -E "^(Name|Version)"
!pip show transformers | grep -E "^(Name|Version)"

# clear_device_cache появился в accelerate 0.33+ — проверим что импорт работает
from accelerate.utils.memory import clear_device_cache
print("\nOK: clear_device_cache импортируется")

In [ ]:
!python -m scripts.train_instruct_lora

## 6. Очистка trainer checkpoints (чтобы архив был компактным)

In [ ]:
!find models/adapters/*/E5_instruct -type d -name 'checkpoint-*' -exec rm -rf {} + 2>/dev/null
!ls -la models/adapters/dostoevsky/E5_instruct/
print('\nРазмеры адаптеров:')
!du -sh models/adapters/*/E5_instruct/

## 7. Архивирование результатов

Kaggle автоматически выставит файлы из `/kaggle/working/` в Output панель ноутбука — там их можно скачать после завершения.

In [ ]:
!cd models/adapters && zip -r /kaggle/working/glyph_e5_instruct_adapters.zip \
    dostoevsky/E5_instruct chekhov/E5_instruct bulgakov/E5_instruct

!cp models/experiment_results_instruct.jsonl /kaggle/working/ 2>/dev/null || true

!ls -la /kaggle/working/*.zip /kaggle/working/*.jsonl 2>/dev/null
print('\nГотово! Файлы доступны в Output панели Kaggle ноутбука.')